## 使用OverWriter绕过reducer，执行默认规则（覆盖）

In [3]:
from operator import add
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.types import Overwrite


# 1、定义状态
class OverAllState(TypedDict):
    # 使用规约，追加
    logs: Annotated[list[str], add]
    id: str


# 2、定义节点
def node_a(state: OverAllState):
    return {
        "logs": ["node_a"],
        "id": "node_a"
    }


def node_b(state: OverAllState):
    # logs 状态中 logs 定义了规约，采用的是追加方式，但Overwrite 支持绕过定义的规约，回到默认的状态更新
    return {
        "logs": Overwrite(["node_b"]),
        "id": "node_b"
    }


def node_c(state: OverAllState):
    return {
        "logs": ["node_c"],
        "id": "node_c"
    }


# 构建图
builder = StateGraph(state_schema=OverAllState)
# 添加节点
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("node_c", node_c)
# 添加边
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", "node_c")
builder.add_edge("node_c", END)
# 获取图
graph = builder.compile()
# 执行图
result = graph.invoke({"logs": ["START"], "id": "START"})
print(result)

{'logs': ['node_b', 'node_c'], 'id': 'node_c'}
